# clikernel

> Connect LLMs to persistent gateway-hosted Jupyter kernels as concise text, over MCP or a plain stream protocol


`clikernel` gives an LLM agent a persistent Python or Luau session. Imports, variables, and results remain available between tool calls. An agent can create a kernel or attach to an existing one, including a user's live solveit kernel, on a local or named remote gateway.

[rustygate](https://github.com/AnswerDotAI/rustygate) hosts the Jupyter kernels and provides their MCP tools. `clikernel` is the conversation-side router: the MCP host launches it over stdio, and it forwards requests to gateways over HTTP. It selects gateways from `gateways.toml`, supplies `startup.py` and `inspectors.py` to Python kernels it creates, and starts a private local gateway when needed. The default kernel is [ipymini](https://github.com/AnswerDotAI/ipymini).

Kernel ownership determines what happens when the conversation ends:

- A session closes kernels it created with `autoclose`, including the automatic kernel used by bare `py` or `lua` and kernels created with `create`'s default settings.
- Attaching with `use_kernel` does not make that session responsible for closing the kernel.
- A kernel created with `autoclose=false` on a resident gateway can outlive the conversation. A later conversation can attach and continue using its state.
- A private child gateway ends with its conversation. Use a resident gateway for kernels that need to survive across conversations.


## Install

```sh
pip install clikernel
```

This installs rustygate and ipymini. No service setup is required for a conversation-local kernel: clikernel starts a private gateway if it cannot find one. To retain kernels across conversations, run a resident gateway, for example through launchd or systemd:

```sh
rustygate --port 8787
```


## Use with an MCP host

Register the stdio server with your MCP host. For Claude Code:

```sh
claude mcp add clikernel -- clikernel-mcp
```

Use `py(code=...)` for Python/IPython or `lua(code=...)` for bundled Luau. Either starts its language's kernel when none is current. There is one current kernel: a language mismatch errors without switching or running the code. IPython magics, including `%%bash`, are Python-only.

For a named kernel, use `create(dlgname="work", language="luau")`. Omit `language` to reuse an existing binding unchanged, or default a new kernel to Python. An explicit language must match an existing binding. The other tools are `list_kernels`, `use_kernel`, `delete_kernel`, `restart`, and `interrupt`; creation, selection, and listing report the language.

These tools forward to rustygate. `list_kernels`, `use_kernel`, and `create` also accept a `host` naming a gateway from `gateways.toml`. One MCP registration can therefore reach multiple machines. Replies retain the gateway's text and image blocks. Python startup and inspectors run only in Python, never Luau, including after restart.

`$CLIKERNEL_HOST` overrides the default gateway URL, `http://127.0.0.1:8787`. If no gateway answers, clikernel uses a private child gateway for the conversation. The ownership rules above determine which kernels close at session end.

Pass `--quiet`, as in `clikernel-mcp --quiet`, to omit startup output from replies. Python startup code still runs.


## Configuration

Three optional files in `$XDG_CONFIG_HOME/clikernel/` configure the router, usually under `~/.config/clikernel/`:

- `startup.py` runs in each Python kernel clikernel creates, with `__file__` set to its path. Its output appears in the reply announcing the kernel unless `--quiet` is set.
- `inspectors.py` installs Python cell inspectors after startup. Define `inspect`, a list named `inspectors`, or both. Each inspector runs once before a cell: a one-argument inspector takes the cell's AST, and a two-argument inspector takes the AST and raw source. Return a string to print a note before the output. Raise the provided `RuleBlock` to block execution. Other exceptions produce a warning and allow the cell to run. See [examples/inspectors.py](examples/inspectors.py).
- `gateways.toml` names remote gateways and configures authentication without putting tokens in tool arguments:

```toml
[gateways.solveit]
url = "https://solveit.example.com/gate"
token_env = "SOLVEIT_TOKEN"
verify = false   # optional: accept a self-signed certificate
```


## The stream protocol

Run `clikernel` as a plain CLI process for clients that read a text stream rather than MCP messages. It uses a delimiter-framed stdin/stdout protocol:

- Input is not echoed.
- Each request gets a `.` acknowledgement.
- A per-process random delimiter marks the end of each response.
- Multiline cells are framed by `--` and the delimiter.

The startup banner supplies the protocol instructions and delimiter. Running `clikernel` without arguments creates a kernel and stops it on exit. `--kernel <id>` attaches to an existing kernel and leaves it running on exit.
